In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import importlib
import src.event_data
importlib.reload(src.event_data)
from src.event_data import generate_event_frames

# -------- Usage --------
event_npz_path = "your path to the .npz file"
rgb_dir = "your path to the RGB images directory"

# Load events
event_npz = np.load(event_npz_path)
t = event_npz["t"].astype(np.int64) 
x = event_npz["x"].astype(np.int16)
y = event_npz["y"].astype(np.int16)
p = event_npz["p"].astype(np.int8)

frames, rgb_imgs, fix_maps, gt_maps = generate_event_frames(
    x, y, p, t,
    rgb_dir=rgb_dir,
    BIN_US=100_000_000,
    n=1,
    plot=False
)


In [ ]:
import numpy as np
import cv2
import importlib
import matplotlib.pyplot as plt
import matplotlib as mpl

import src.evaluate_saliency as es
importlib.reload(es)

import src.saliency_v1
importlib.reload(src.saliency_v1)

from src.saliency_v1 import spectral_residual_saliency, compute_bms_saliency

import src.saliency_v2
importlib.reload(src.saliency_v2)
from src.saliency_v2 import spectral_residual_saliency_v2, compute_bms_saliency_v2
from src.evaluate_saliency import KLdiv, AUC_Judd, NSS, sim, CC

import src.attention_saliency
importlib.reload(src.attention_saliency)
from src.attention_saliency import spectral_residual_saliency_nn, compute_bms_saliency_nn



# ==========================================================
# LOOP THROUGH GENERATED FRAMES
# ==========================================================
for n in range(len(frames)):

    # -----------------------------
    # Event frame → grayscale RGB
    # -----------------------------
    event_frame = frames[n].astype(np.float32)

    ev_gray = (event_frame - event_frame.min()) / (
        event_frame.max() - event_frame.min() + 1e-8
    )
    ev_gray_255 = (ev_gray * 255).astype(np.uint8)
    ev_rgb = np.stack([ev_gray_255] * 3, axis=-1)

    # -----------------------------
    # Saliency on EVENT frame
    # -----------------------------
    # sr_map_on_event = spectral_residual_saliency(ev_rgb)
    # bms_map_on_event = compute_bms_saliency(ev_rgb)

    sr_map_on_event = spectral_residual_saliency_nn(ev_rgb)
    bms_map_on_event = compute_bms_saliency_nn(ev_rgb)

    # sr_map_on_event = spectral_residual_saliency_v2(ev_rgb)
    # bms_map_on_event = compute_bms_saliency_v2(ev_rgb)

    # -----------------------------
    # Original RGB image
    # -----------------------------
    img_orig = rgb_imgs[n]
    if img_orig.max() <= 1.0:
        img_orig = (img_orig * 255).astype(np.uint8)

    sr_map_on_rgb = spectral_residual_saliency(img_orig)
    bms_map_on_rgb = compute_bms_saliency(img_orig)

    # -----------------------------
    # Fixation & GT maps (ALREADY LOADED)
    # -----------------------------
    fixation_map = fix_maps[n]
    gtsal_map = gt_maps[n]

    if fixation_map is None or gtsal_map is None:
        print(f"[Skip frame {n}] Missing fixation or GT map")
        continue

    fixation_map = (fixation_map > 128).astype(np.uint8)

    gtsal_map = gtsal_map.astype(np.float32)
    gtsal_map = (gtsal_map - gtsal_map.min()) / (
        gtsal_map.max() - gtsal_map.min() + 1e-8
    )

    H, W = fixation_map.shape

    # -----------------------------
    # Resize saliency maps
    # -----------------------------
    sr_map_event_resized = cv2.resize(sr_map_on_event, (W, H))
    bms_map_event_resized = cv2.resize(bms_map_on_event, (W, H))
    sr_map_rgb_resized = cv2.resize(sr_map_on_rgb, (W, H))
    bms_map_rgb_resized = cv2.resize(bms_map_on_rgb, (W, H))

    gtsal_map_resized = cv2.resize(gtsal_map, (W, H))

    -----------------------------
    Metrics – EVENT
    -----------------------------
    auc_sr  = AUC_Judd(sr_map_event_resized, fixation_map)
    nss_sr  = NSS(sr_map_event_resized, fixation_map)
    kl_sr   = KLdiv(sr_map_event_resized, gtsal_map_resized)
    cc_sr   = CC(sr_map_event_resized, gtsal_map_resized)
    sim_sr  = sim(sr_map_event_resized, gtsal_map_resized)

    auc_bms = AUC_Judd(bms_map_event_resized, fixation_map)
    nss_bms = NSS(bms_map_event_resized, fixation_map)
    kl_bms  = KLdiv(bms_map_event_resized, gtsal_map_resized)
    cc_bms  = CC(bms_map_event_resized, gtsal_map_resized)
    sim_bms = sim(bms_map_event_resized, gtsal_map_resized)

    # -----------------------------
    # Metrics – RGB
    # -----------------------------
    auc_sr_rgb  = AUC_Judd(sr_map_rgb_resized, fixation_map)
    nss_sr_rgb  = NSS(sr_map_rgb_resized, fixation_map)
    kl_sr_rgb   = KLdiv(sr_map_rgb_resized, gtsal_map_resized)
    cc_sr_rgb   = CC(sr_map_rgb_resized, gtsal_map_resized)
    sim_sr_rgb  = sim(sr_map_rgb_resized, gtsal_map_resized)

    auc_bms_rgb = AUC_Judd(bms_map_rgb_resized, fixation_map)
    nss_bms_rgb = NSS(bms_map_rgb_resized, fixation_map)
    kl_bms_rgb  = KLdiv(bms_map_rgb_resized, gtsal_map_resized)
    cc_bms_rgb  = CC(bms_map_rgb_resized, gtsal_map_resized)
    sim_bms_rgb = sim(bms_map_rgb_resized, gtsal_map_resized)

    # -----------------------------
    # Print results
    # -----------------------------
    print(f"\nFrame {n}")
    print(f" SR EVENT  | AUC {auc_sr:.3f} | NSS {nss_sr:.3f} | KL {kl_sr:.3f} | CC {cc_sr:.3f} | SIM {sim_sr:.3f}")
    print(f" BMS EVENT | AUC {auc_bms:.3f} | NSS {nss_bms:.3f} | KL {kl_bms:.3f} | CC {cc_bms:.3f} | SIM {sim_bms:.3f}")
    print(f" SR RGB    | AUC {auc_sr_rgb:.3f} | NSS {nss_sr_rgb:.3f} | KL {kl_sr_rgb:.3f} | CC {cc_sr_rgb:.3f} | SIM {sim_sr_rgb:.3f}")
    print(f" BMS RGB   | AUC {auc_bms_rgb:.3f} | NSS {nss_bms_rgb:.3f} | KL {kl_bms_rgb:.3f} | CC {cc_bms_rgb:.3f} | SIM {sim_bms_rgb:.3f}")

    # -----------------------------
    # Visualization
    # -----------------------------
    fig, axes = plt.subplots(1, 7, figsize=(18, 4))
    axes[0].imshow(ev_gray, cmap="gray"); axes[0].set_title("Event")
    axes[1].imshow(sr_map_event_resized, cmap="gray"); axes[1].set_title("SR on Event",size=12)
    axes[2].imshow(bms_map_event_resized, cmap="gray"); axes[2].set_title("BMS on Event",size=12)
    axes[3].imshow(img_orig); axes[3].set_title("RGB",size=12)
    axes[4].imshow(sr_map_rgb_resized, cmap="gray"); axes[4].set_title("SR on RGB",size=12)
    axes[5].imshow(bms_map_rgb_resized, cmap="gray"); axes[5].set_title("BMS on RGB",size=12)
    axes[6].imshow(gtsal_map_resized, cmap="gray"); axes[6].set_title("GT",size=12)

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
